# Whisper LoRA Inference & Evaluation — Nepali-English Code-Mixed
This notebook loads the fine-tuned LoRA checkpoint from your previous run and evaluates its performance (WER, CER, Language-specific WER) on a subset of the data.

In [ ]:
!pip install -q transformers datasets accelerate peft librosa soundfile pandas jiwer
!pip uninstall -y -q torchao || true

In [ ]:
import os
import re
import torch
import librosa
import pandas as pd
import jiwer
import numpy as np
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# --- 1. SET PATHS ---
CHECKPOINT_DIR = "/kaggle/input/notebooks/leo17messi/whisper-largecodemix/outputs/best_checkpoint"
BASE_MODEL = "openai/whisper-large-v3"

# --- 2. LOAD PROCESSOR ---
print("Loading processor...")
try:
    processor = WhisperProcessor.from_pretrained(CHECKPOINT_DIR)
except Exception:
    processor = WhisperProcessor.from_pretrained(BASE_MODEL)

# --- 3. LOAD MODEL + LoRA ADAPTER ---
print("Loading base model...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("Loading LoRA adapter and merging weights...")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model = model.merge_and_unload()  # Merge LoRA weights into base model for faster inference
model = model.to(device)
model.eval()

# Set forced decoder IDs for Nepali
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ne", task="transcribe")
model.generation_config.suppress_tokens = []

print("Model ready for inference!")

In [ ]:
DEVANAGARI_START = 0x0900
DEVANAGARI_END = 0x097F

def detect_language(token: str) -> str:
    if not token or not token.strip(): return "other"
    token = token.strip()
    if any(DEVANAGARI_START <= ord(c) <= DEVANAGARI_END for c in token):
        return "ne"
    cleaned = re.sub(r'[.,!?;:\'"\-()\[\]{}«»…]', '', token)
    if not cleaned: return "other"
    if cleaned.replace('.', '').replace('-', '').isdigit(): return "num"
    if all(c.isascii() and c.isalpha() for c in cleaned): return "en"
    return "other"

def filter_text_by_language(text: str, lang: str) -> str:
    tokens = text.split()
    filtered = [t for t in tokens if detect_language(t) == lang]
    return " ".join(filtered)

def compute_wer(reference: str, hypothesis: str) -> float:
    if not reference and not hypothesis: return 0.0
    if not reference: return 1.0
    if not hypothesis: return 1.0
    return jiwer.wer(reference, hypothesis)

def compute_cer(reference: str, hypothesis: str) -> float:
    if not reference and not hypothesis: return 0.0
    if not reference: return 1.0
    if not hypothesis: return 1.0
    return jiwer.cer(reference, hypothesis)

def compute_wer_nepali(reference: str, hypothesis: str) -> float:
    ref_ne = filter_text_by_language(reference, "ne")
    hyp_ne = filter_text_by_language(hypothesis, "ne")
    if not ref_ne and not hyp_ne: return 0.0
    if not ref_ne: return float('inf')
    return jiwer.wer(ref_ne, hyp_ne)

def compute_wer_english(reference: str, hypothesis: str) -> float:
    ref_en = filter_text_by_language(reference, "en")
    hyp_en = filter_text_by_language(hypothesis, "en")
    if not ref_en and not hyp_en: return 0.0
    if not ref_en: return float('inf')
    return jiwer.wer(ref_en, hyp_en)


In [ ]:
def transcribe_audio(audio_path):
    # Load and resample audio to 16kHz
    array, sr = librosa.load(audio_path, sr=16000, mono=True)
    
    # Extract features
    inputs = processor.feature_extractor(
        array, 
        sampling_rate=16000, 
        return_tensors="pt"
    ).to(device)
    inputs["input_features"] = inputs["input_features"].to(model.dtype)
    
    # Generate prediction
    with torch.no_grad():
        generated_ids = model.generate(
            input_features=inputs["input_features"],
            language="ne",
            task="transcribe",
            max_length=225,
        )
        
    # Decode
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return transcription.strip()

In [ ]:
# --- EVALUATE ON A SUBSET ---
CSV_PATH = "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle1.csv"
AUDIO_DIR = "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched/kaggle_upload/audios_segment"

NUM_SAMPLES = 100  # Adjust this to evaluate on more/less data

df = pd.read_csv(CSV_PATH, on_bad_lines='skip', engine='python')
# Setting a seed so the subset is consistent across runs
samples = df.sample(min(NUM_SAMPLES, len(df)), random_state=42) 

print("="*70)
print(f"RUNNING EVALUATION ON {len(samples)} SAMPLES")
print("="*70)

total_wer = 0.0
total_cer = 0.0
total_wer_ne = 0.0
total_wer_en = 0.0
ne_count = 0
en_count = 0

for idx, row in tqdm(samples.iterrows(), total=len(samples), desc="Evaluating"):
    filename = str(row.iloc[0]).strip()
    reference_text = str(row.iloc[1]).strip()
    
    if not os.path.isabs(filename):
        audio_path = os.path.join(AUDIO_DIR, os.path.basename(filename))
    else:
        audio_path = filename
        
    if not os.path.exists(audio_path):
        continue
        
    prediction = transcribe_audio(audio_path)
    
    wer = compute_wer(reference_text, prediction)
    cer = compute_cer(reference_text, prediction)
    wer_ne = compute_wer_nepali(reference_text, prediction)
    wer_en = compute_wer_english(reference_text, prediction)
    
    total_wer += wer
    total_cer += cer
    if wer_ne != float('inf'):
        total_wer_ne += wer_ne
        ne_count += 1
    if wer_en != float('inf'):
        total_wer_en += wer_en
        en_count += 1
        
    # Print first 5 just to see some outputs
    if idx < samples.index[5]:
        print(f"\n🎯 Ref : {reference_text}")
        print(f"🤖 Pred: {prediction}")
        print(f"   WER: {wer:.3f} | CER: {cer:.3f}")

n = len(samples)
print("\n" + "="*70)
print("FINAL EVALUATION METRICS")
print("="*70)
print(f"Overall WER      : {total_wer / n if n > 0 else 0:.4f}")
print(f"Overall CER      : {total_cer / n if n > 0 else 0:.4f}")
print(f"Nepali WER       : {total_wer_ne / ne_count if ne_count > 0 else 0:.4f} (across {ne_count} samples with Nepali)")
print(f"English WER      : {total_wer_en / en_count if en_count > 0 else 0:.4f} (across {en_count} samples with English)")
print("="*70)